**Instructions**: If the name of this notebook contains your ID number then write your code in this notebook. If you write your code somewhere else (possibly in a copy of this notebook) then those code will not be received or submitted. Do not create a copy at all to avoid mistakes. Read all the comments and write appropriate code near any place that says `# YOUR CODE HERE`. When the daily evaluation tasks are revealed add their solutions at the end of this notebook. After the end of the evaluation, you will not have any edit access to this notebook.

---
# CSE330 Lab: Linear System Solving
---

## Importing Necessary Libraries

In [1]:
import numpy as np
np.set_printoptions(formatter={'all': lambda x: f'{x:f}'})

## Part 1: Using Inverse Matrix

Suppose, we have a linear system

\begin{align}
&a_{11} x_1 + a_{12} x_2 + \cdots  + a_{1n} x_n = b_1 \tag{1}\\
&a_{21} x_1 + a_{22} x_2 + \cdots + a_{2n} x_n = b_2 \tag{2}\\
&\cdots \tag{...}\\
&a_{n1} x_1 + a_{n2} x_2 + \cdots + a_{nn} x_n = b_n \tag{n}\\
\end{align}

It is convenient to express this system in the
matrix form

\begin{align}
Ax = b \tag{7.1}
\end{align}

where $A$ is an $n \times n$ square matrix with elements $a_{ij}$, and $x$, $b$ are $n \times 1$ vectors.

Note that this system has a unique solution given by $x = A^{-1}b$ iff $A$ is non-singular.

### Task 1 - 2 Marks
Complete the code below to solve $Ax = b$ for $x$

In [2]:
def get_result_by_inverse_matrix(A, b):
    # n = len(b) = number of unknowns.
    # b is a 1D numpy array of size n.
    # A is a 2D numpy array of size n by n.
    # First check if the determinant of A is non-zero.
    # Return None if there is no unique solution to the system.
    # If there is a unique solution then return a numpy array of size n.
    # You may use functions such as linalg.det(), linalg.inv() from numpy.

    det_A = np.linalg.det(A)
    if np.isclose(det_A, 0.0):
        return None

    inv_A = np.linalg.inv(A)
    x = inv_A @ b
    return x

In [3]:
def check_result_by_inverse_matrix(data_A, data_b):
    result = get_result_by_inverse_matrix(data_A.copy(), data_b.copy())
    print(result)
    if np.linalg.det(data_A) == 0:
        assert result == None
    else:
        np.testing.assert_array_almost_equal(np.dot(data_A, result), data_b)

data_A = np.array([[1, 2, 1],
                   [1, -2, 2],
                   [2, 12, -2]])
data_b = np.array([0, 4, 4])
check_result_by_inverse_matrix(data_A, data_b)

data_A = np.array([[1, 2, 1],
                   [1, -2, 2],
                   [2, 0, 3]])
data_b = np.array([0, 4, 4])
check_result_by_inverse_matrix(data_A, data_b)

data_A = np.array([[1, 1, 1],
                   [0, 1, 1],
                   [1, 2, 2]])
data_b = np.array([1, 2, 3])
check_result_by_inverse_matrix(data_A, data_b)

data_A = np.array([[0, 0, 0, 1],
                   [0, 3, 2, 1],
                   [4, 3, 2, 1],
                   [0, 0, 2, 1]])
data_b = np.array([1, 3, 4, 2])
check_result_by_inverse_matrix(data_A, data_b)

[11.000000 -2.500000 -6.000000]
None
None
[0.250000 0.333333 0.500000 1.000000]


## Part 2: Gaussian Elimination

Gaussian elimination uses elementary row operations to transform the system $Ax = b$ to upper triangular form $Ux = y$.

Elementary row operations include swapping rows and adding multiples of one row to another. These operations do not change the solution $x$, but change the matrix $A$ and the vector $b$.

**Algorithm for Gaussian Elimination**

For each row $i$ in the increasing order, we perform row operations into the matrix $A$ (and the vector $b$ at the same time) such that after the operations, $a_{ii}$ becomes non-zero and $a_{ki}$ become zeros for each $k>i$.

To make $a_{ii}$ non-zero, we first check if it is already non-zero. If it is already non-zero then we are done. If it is zero then we find any $k>i$ such that $a_{ki}$ is non-zero and swap the $k$'th row with the $i$'th row (in both the matrix $A$ and the vector $b$). If no such $k$ exists then $det(A)=0$ and we stop since there is no unique solution to the system.

Once we have a non-zero $a_{ii}$, for each $k>i$ we find the row multipliers.

\begin{align}
m_{ki} = \frac{a_{ki}}{a_{ii}} \tag{7.2}
\end{align}

We use these row multipliers to change $a_{ik}$ into zeros for all $k>i$.

\begin{align}
a_{ki}=a_{ki}-m_{ki}a_{ii}=0 \tag{7.3}
\end{align}

But to keep the solution unchanged, we need to do the same operation for each entry in the $k$'th row (in both the matrix $A$ and the vector $b$).

\begin{align}
a_{kj} &= a_{kj}-m_{ki}a_{ij} \tag{7.4}\\
b_{k} &= b_{k}-m_{ki}b_{i} \tag{7.5}
\end{align}

Since we always do these row operations on both the matrix $A$ and the vector $b$, it is helpful to start by combining these matrices to form an augmented matrix.

\begin{bmatrix}
a_{11} & a_{12} & \cdots & a_{1n} & b_{1}\\
a_{21} & a_{22} & \cdots & a_{2n} & b_{2}\\
\vdots & \vdots & \ddots & \vdots & \vdots\\
a_{n1} & a_{n2} & \cdots & a_{nn} & b_{n}\\
\end{bmatrix}

After performing the necessary row operations on the augmented matrix, we find the upper triangular matrix $𝑈$, augmented with $y$.

\begin{bmatrix}
u_{11} & u_{12} & \cdots & u_{1n} & y_{1}\\
0 & u_{22} & \cdots & u_{2n} & y_{2}\\
\vdots & \vdots & \ddots & \vdots & \vdots\\
0 & \cdots & 0 & u_{nn} & y_{n}\\
\end{bmatrix}

After generating the upper triangular matrix, **Backward Substitution** is used to get the solution. For any $n \times n$ upper triangular system, $Ux = y$, the solution is:

\begin{align}
x_i = \frac{y_i-\sum_{j=i+1}^{n}u_{ij}x_j}{u_{ii}} \tag{7.6}
\end{align}

Since we need the values of $x_j$ with $j>i$ to find $x_i$, we find $x_i$ in the decreasing order of $i$. Note that this works only when $det(𝑈) \neq 0$ which is true iff $det(A) \neq 0$.

### Task 2 - 2 Marks
Complete the code below to solve for $x$

In [4]:
def get_result_gaussian_elimination(n, A):
    # n is the number of unknowns.
    # A is a 2D numpy array of size n by (n+1).
    # First check if the solution to the system is unique.
    # Return None if there is no unique solution to the system.
    # If there is a unique solution then apply Gaussian Elimination,
    # followed by Back Substitution to find and return a numpy array of size n.

    A = A.astype(float, copy=False)

    # Forward elimination with partial pivoting
    for i in range(n):
        pivot_row = i + int(np.argmax(np.abs(A[i:, i])))
        if np.isclose(A[pivot_row, i], 0.0):
            return None

        if pivot_row != i:
            A[[i, pivot_row]] = A[[pivot_row, i]]

        for k in range(i + 1, n):
            multiplier = A[k, i] / A[i, i]
            A[k, i:] = A[k, i:] - multiplier * A[i, i:]

    # Back substitution
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        if np.isclose(A[i, i], 0.0):
            return None
        rhs = A[i, -1] - np.dot(A[i, i + 1:n], x[i + 1:n])
        x[i] = rhs / A[i, i]

    return x

In [ ]:
def check_result_gaussian_elimination(data_A):
    result = get_result_gaussian_elimination(data_A.shape[0], data_A.copy())
    print(result)
    if np.linalg.det(data_A[:, :-1]) == 0:
        assert result == None
    else:
        np.testing.assert_array_almost_equal(np.dot(data_A[:, :-1], result),
                                             data_A[:, -1])

data_A = np.array([[1, 2, 1, 0],
                   [1, -2, 2, 4],
                   [2, 12, -2, 4]])
check_result_gaussian_elimination(data_A)

data_A = np.array([[0, 1, 1, 5],
                   [1, -2, 2, 4],
                   [2, 12, -2, 4]])
check_result_gaussian_elimination(data_A)

data_A = np.array([[1, 1, 1, 1],
                   [0, 1, 1, 2],
                   [1, 2, 2, 3]])
check_result_gaussian_elimination(data_A)

data_A = np.array([[0, 0, 0, 1, 1],
                   [0, 3, 2, 1, 3],
                   [4, 3, 2, 1, 4],
                   [0, 0, 2, 1, 2]])
check_result_gaussian_elimination(data_A)

# Daily Evaluation - 4 Marks